In [ ]:
!git clone https://github.com/PSY222/Colorinsight.git

Cloning into 'Colorinsight'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 86 (delta 6), reused 5 (delta 0), pack-reused 57 (from 1)
Receiving objects: 100% (86/86), 109.40 MiB | 13.97 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (41/41), done.


In [ ]:
!pip install validators

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 361.8 kB/s eta 0:00:00


In [ ]:
%cd /content/Colorinsight/facer

/content/Colorinsight/facer


In [ ]:
import argparse
import glob
import os
import os.path as osp
import random
from collections import Counter

import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from skimage.filters import gaussian

import facer
# from model import BiSeNet


def get_rgb_codes(path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    image = facer.hwc2bchw(facer.read_hwc(path)).to(device=device)
    face_detector = facer.face_detector('retinaface/mobilenet', device=device)
    with torch.inference_mode():
        faces = face_detector(image)

    face_parser = facer.face_parser('farl/lapa/448', device=device)
    with torch.inference_mode():
        faces = face_parser(image, faces)

    seg_logits = faces['seg']['logits']
    seg_probs = seg_logits.softmax(dim=1)  # nfaces x nclasses x h x w
    seg_probs = seg_probs.cpu() #if you are using GPU

    tensor = seg_probs.permute(0, 2, 3, 1)
    tensor = tensor.squeeze().numpy()

    llip = tensor[:, :, 7]
    ulip = tensor[:,:,9]
    lips = llip+ulip
    binary_mask = (lips >= 0.5).astype(int)

    sample = cv2.imread(path)
    img = cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)

    indices = np.argwhere(binary_mask)   #binary mask location extraction
    rgb_codes = img[indices[:, 0], indices[:, 1], :] #RGB color extraction by pixels
    return rgb_codes

def filter_lip_random(rgb_codes,randomNum=40):
    blue_condition = (rgb_codes[:, 2] <= 227)
    red_condition = (rgb_codes[:, 0] >= 97)
    filtered_rgb_codes = rgb_codes[blue_condition & red_condition]
    random_index = np.random.randint(0,filtered_rgb_codes.shape[0],randomNum)
    random_rgb_codes = filtered_rgb_codes[random_index]
    return random_rgb_codes


def calc_dis(rgb_codes):
    spring = [[253,183,169],[247,98,77],[186,33,33]]
    summer = [[243,184,202],[211,118,155],[147,70,105]]
    autum = [[210,124,110],[155,70,60],[97,16,28]]
    winter = [[237,223,227],[177,47,57],[98,14,37]]

    res = []
    for i in range(len(rgb_codes)):
      sp = np.inf
      su = np.inf
      au = np.inf
      win = np.inf
      for j in range(3):
        sp = min(sp, np.linalg.norm(rgb_codes[i] - spring[j]))
        su = min(su, np.linalg.norm(rgb_codes[i]- summer[j]))
        au = min(au, np.linalg.norm(rgb_codes[i] - autum[j]))
        win = min(win, np.linalg.norm(rgb_codes[i] - winter[j]))

      min_type = min(sp, su, au, win)
      if min_type == sp:
        ctype = "sp"
      elif min_type == su:
        ctype = "su"
      elif min_type == au:
        ctype = "au"
      elif min_type == win:
        ctype = "win"

      res.append(ctype)
    return res


def save_skin_mask(img_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    image = facer.hwc2bchw(facer.read_hwc(img_path)).to(device=device)  # image: 1 x 3 x h x w
    face_detector = facer.face_detector('retinaface/mobilenet', device=device)

    with torch.inference_mode():
      faces = face_detector(image)

    image = facer.hwc2bchw(facer.read_hwc(img_path)).to(device=device)
    face_parser = facer.face_parser('farl/lapa/448', device=device)
    with torch.inference_mode():
      faces = face_parser(image, faces)

    seg_logits = faces['seg']['logits']
    seg_probs = seg_logits.softmax(dim=1)
    seg_probs = seg_probs.cpu() #if you are using GPU
    tensor = seg_probs.permute(0, 2, 3, 1)
    tensor = tensor.squeeze().numpy()

    face_skin = tensor[:, :, 1]
    binary_mask = (face_skin >= 0.5).astype(int)

    sample = cv2.imread(img_path)
    img = cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)
    masked_image = np.zeros_like(img)
    try:
      masked_image[binary_mask == 1] = img[binary_mask == 1]
      masked_image = cv2.cvtColor(masked_image,cv2.COLOR_BGR2RGB)
      cv2.imwrite("temp.jpg" , masked_image)
    except:
      print("error occurred")

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn as nn

def get_season(img):
    model = models.resnet18(pretrained=True)
    num_classes = 4
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # load saved state dictionary
    state_dict = torch.load('best_model_resnet_ALL.pth', map_location=torch.device('cpu'))

    # create a new model with the correct architecture
    new_model = models.resnet18(pretrained=True)
    new_model.fc = nn.Linear(in_features, num_classes)
    new_model.load_state_dict(state_dict)

    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    image = Image.open(img).convert('RGB')
    image = transform(image).unsqueeze(0)

    new_model.eval()

    with torch.no_grad():
        output = new_model(image)
    pred_index = output.argmax().item()
    print("Decided color: ",pred_index)
    return pred_index


In [ ]:
# test.py


from collections import Counter

# === CONFIG ===
TEST_IMG = "/content/360_F_570574724_HWfki1q3XZt9WzVlCcQujOV5Jxe8UBG1.jpg"  # use your test image here

# === STEP 1: Make skin mask ===
print("Generating skin mask...")
save_skin_mask(TEST_IMG)

print("Skin mask saved as temp.jpg")

# === STEP 2: Run season classifier ===
print("Running season classifier...")
result = get_season("temp.jpg")
print(f"Predicted season index: {result}")

# === STEP 3: Extract lip RGB codes ===
print("Extracting lip RGB codes...")
rgb_codes = get_rgb_codes(TEST_IMG)
print(f"Found {len(rgb_codes)} lip pixels.")

# === STEP 4: Filter + sample lip RGB codes ===
print("Filtering & sampling lip RGB codes...")
samples = filter_lip_random(rgb_codes, randomNum=40)
print(f"Sampled {len(samples)} RGB codes.")

# === STEP 5: Calculate closest palette ===
print("Calculating distances...")
dis = calc_dis(samples)

counts = Counter(dis)
print("Counts by type:", counts)

dominant = counts.most_common(1)[0][0]
print(f"Dominant lip palette: {dominant}")


SEASON_COLOR_MAP = {
    'sp': ["Coral", "Pink", "Yellow", "Light Green", "Light Blue", "Peach"],   # Spring
    'su': ["Light Pink", "Purple", "Blue", "Mint Green", "Grey", "Rose"],      # Summer
    'au': ["Beige", "Brown", "Olive", "Dark Green", "Dark Orange", "Mustard"], # Autumn
    'wi': ["Dark Red", "Dark Green", "Dark Blue", "Charcoal", "Plum", "Teal"]  # Winter
}

result=SEASON_COLOR_MAP[dominant]
print("------------------------Color Codes------------------------")
for i in range(6):

  print(result[i])


Generating skin mask...


Downloading: "https://github.com/elliottzheng/face-detection/releases/download/0.0.1/mobilenet0.25_Final.pth" to /root/.cache/torch/hub/checkpoints/mobilenet0.25_Final.pth
100%|██████████| 1.71M/1.71M [00:00<00:00, 27.2MB/s]
Downloading: "https://github.com/FacePerceiver/facer/releases/download/models-v1/face_parsing.farl.lapa.main_ema_136500_jit191.pt" to /root/.cache/torch/hub/checkpoints/face_parsing.farl.lapa.main_ema_136500_jit191.pt
100%|██████████| 617M/617M [00:08<00:00, 76.3MB/s]
/usr/local/lib/python3.11/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Skin mask saved as temp.jpg
Running season classifier...


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


Decided color:  3
Predicted season index: 3
Extracting lip RGB codes...
Found 441 lip pixels.
Filtering & sampling lip RGB codes...
Sampled 40 RGB codes.
Calculating distances...
Counts by type: Counter({'au': 28, 'su': 9, 'sp': 3})
Dominant lip palette: au
------------------------Color Codes------------------------
Beige
Brown
Olive
Dark Green
Dark Orange
Mustard
